# Balance Sheet & Working Capital Analysis (CFA Level 1)

A from-scratch exploration of balance sheet structure, inventory cost flow methods, depreciation, and working capital management.

**Prerequisites:** Introduction to Financial Statements, Income Statement Analysis.

**Outline**
1. The classified balance sheet
2. Setup
3. Current vs non-current classification
4. Inventory cost flow methods — FIFO, LIFO, weighted average
5. LIFO reserve and LIFO liquidation
6. Depreciation methods
7. Impairment of long-lived assets
8. Working capital and the cash conversion cycle
9. Liquidity ratios
10. Off-balance-sheet items
11. References

This notebook covers the most analytically rich areas of balance sheet analysis: how inventory accounting methods create divergent financial pictures from identical economic transactions, how depreciation choices affect reported profitability, and how working capital efficiency determines how much cash a growing business needs to fund its operations.

---
## 1. The Classified Balance Sheet

The balance sheet (or **Statement of Financial Position**) presents a company's financial position **at a single point in time**. It is organised around the accounting equation:

$$
\text{Assets} = \text{Liabilities} + \text{Equity}
$$

A **classified** balance sheet groups items by liquidity (how quickly they convert to cash) and maturity (when obligations come due):

| **Assets** | **Liabilities** | **Equity** |
|-----------|----------------|-----------|
| *Current Assets* | *Current Liabilities* | Share capital |
| Cash & equivalents | Accounts payable | Additional paid-in capital |
| Short-term investments | Short-term debt | Retained earnings |
| Accounts receivable | Current portion of LT debt | Treasury stock |
| Inventory | Accrued expenses | Accumulated OCI |
| Prepaid expenses | Deferred revenue | |
| *Non-Current Assets* | *Non-Current Liabilities* | |
| Property, plant & equipment | Long-term debt | |
| Intangible assets | Pension obligations | |
| Goodwill | Deferred tax liabilities | |

> **Key Concept:** The balance sheet is a *stock* variable — a snapshot at an instant. The income statement and cash flow statement are *flow* variables covering a period. When computing ratios that mix stocks and flows (e.g. ROA), use the **average** of beginning and ending balance sheet values.

### IFRS vs US GAAP presentation

| Feature | IFRS | US GAAP |
|---------|------|---------|
| Asset ordering | Non-current first (common, not required) | Current first |
| Minimum line items | Specified in IAS 1 | Not specified |
| Revaluation | Permitted (fair value model) | Not permitted (historical cost) |
| Current/non-current distinction | Required (unless liquidity order is more informative) | Required |

> **CFA Exam Tip:** IFRS balance sheets often list non-current assets *above* current assets (reverse of US GAAP convention). The economics are identical; only the presentation order differs.

### Reading a balance sheet — what to look for first

Experienced analysts develop a systematic approach to reading balance sheets. Here are the key questions to ask:

1. **What is the asset composition?** — Is the company asset-light (mostly cash and receivables, like a software company) or asset-heavy (mostly PP&E, like a utility)?
2. **What is the capital structure?** — How much is funded by debt vs equity? Compute Debt/Equity and Debt/Assets ratios immediately.
3. **What is the liquidity position?** — Is current assets > current liabilities? By how much? Is the current ratio improving or deteriorating?
4. **Are there large intangible assets?** — Significant goodwill may indicate acquisition-driven growth. Intangibles can be written down, creating sudden balance sheet shocks.
5. **What is the quality of equity?** — Is retained earnings growing (the company is profitable and retaining earnings) or negative (accumulated losses)?

> **Key Concept:** The balance sheet reveals the *structural choices* management has made about how to finance the business. A company with 80% debt financing has made a fundamentally different risk-return trade-off than one with 20% debt — and this choice affects everything from cost of capital to financial flexibility to bankruptcy risk.

---
## 2. Setup

This notebook uses synthetic but realistic financial data to demonstrate each concept. The inventory and depreciation examples use small, traceable numbers so you can verify every calculation by hand — a crucial skill for the CFA exam, where no calculators with spreadsheet functions are permitted.

We follow the same conventions as the rest of the series: NumPy for computation, matplotlib for visualisation, and the steelblue/coral/seagreen/gold colour palette for consistency.

In [ ]:
%matplotlib inline
import numpy as np
from scipy import stats, optimize, linalg
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

SEED = 42
rng = np.random.default_rng(SEED)

ATOL = 1e-10
RTOL = 1e-6

PRIMARY   = 'steelblue'
SECONDARY = 'coral'
TERTIARY  = 'seagreen'
ACCENT    = 'gold'
plt.rcParams.update({'figure.figsize': (10, 6), 'font.size': 12, 'axes.grid': True, 'grid.alpha': 0.3})

---
## 3. Current vs Non-Current Classification

### Current assets

An asset is **current** if it is expected to be realised (converted to cash), consumed, or sold within **one year** or the company's **operating cycle**, whichever is longer.

The operating cycle is the time from purchasing inventory to collecting cash from customers:

$$
\text{Operating Cycle} = \text{Days Inventory Outstanding} + \text{Days Sales Outstanding}
$$

For most companies the operating cycle is less than one year, but some industries (e.g. shipbuilding, real estate development) may have multi-year cycles.

**Common current assets (in order of liquidity):**
1. **Cash and cash equivalents** — currency, demand deposits, money market funds (maturity < 3 months)
2. **Short-term investments** — trading securities, available-for-sale securities
3. **Accounts receivable** — amounts owed by customers (net of allowance for doubtful accounts)
4. **Inventory** — raw materials, work-in-progress, finished goods
5. **Prepaid expenses** — rent, insurance paid in advance

### Non-current assets

Everything else — expected to provide economic benefit beyond one year:
* **Property, Plant & Equipment (PP&E)** — land, buildings, machinery (reported net of accumulated depreciation)
* **Intangible assets** — patents, trademarks, customer lists (definite life: amortised; indefinite life: tested for impairment)
* **Goodwill** — premium paid in acquisition above fair value of identifiable net assets (never amortised under IFRS/GAAP, tested for impairment annually)
* **Investments** — equity method investments, joint ventures, long-term bonds held to maturity

> **Common Mistake:** Goodwill is not amortised under current IFRS or US GAAP — it is tested for impairment annually. However, this is an active debate, and the IASB has considered reintroducing amortisation. Check the exam date's applicable standard.

### Current liabilities

Obligations due within one year or the operating cycle:
* **Accounts payable** — amounts owed to suppliers
* **Short-term debt** — bank lines of credit, commercial paper
* **Current portion of long-term debt** — the principal repayment due within 12 months
* **Accrued expenses** — salaries, utilities, interest that has been incurred but not yet paid
* **Deferred revenue** — payments received for goods/services not yet delivered

### Accounts receivable and the allowance method

Not all receivables will be collected. Companies estimate the portion that will default using the **allowance method**:

$$\text{Net Receivables} = \text{Gross Receivables} - \text{Allowance for Doubtful Accounts}$$

The allowance is an estimate — and therefore a judgment call that affects both the balance sheet (net receivables) and the income statement (bad debt expense). An overly optimistic allowance understates expenses and overstates assets.

Two approaches to estimating the allowance:
* **Percentage of sales method** — estimates bad debt expense as a fixed percentage of revenue (focuses on the income statement)
* **Aging of receivables method** — categorises receivables by age and applies increasing default rates to older buckets (focuses on the balance sheet)

> **CFA Exam Tip:** If a company *reduces* its allowance for doubtful accounts without a clear improvement in customer quality, this is a red flag — it artificially boosts income by reducing bad debt expense. Watch for the allowance-to-receivables ratio declining over time.

### Intangible assets — finite vs indefinite life

The treatment of intangible assets depends on whether their useful life is determinable:

| Category | Examples | Treatment |
|----------|----------|-----------|
| **Finite life** | Patents (20 years), software (3-5 years), customer lists | Amortised over useful life (like depreciation) |
| **Indefinite life** | Trademarks, broadcast licences | Not amortised; tested for impairment annually |
| **Goodwill** | Premium paid in acquisitions | Not amortised; tested for impairment annually |

> **Key Concept:** Internally generated intangibles (brand value, customer loyalty, workforce quality) are generally **not recognised** on the balance sheet — only *purchased* intangibles from acquisitions are recorded. This creates a systematic understatement of assets for companies that have built (rather than bought) their intangible value, and is one of the most significant limitations of balance sheet analysis.

---
## 4. Inventory Cost Flow Methods

When a company buys inventory at different prices over time, it must choose a **cost flow assumption** to determine:
1. **COGS** (income statement) — the cost assigned to units *sold*
2. **Ending inventory** (balance sheet) — the cost assigned to units *remaining*

The three methods are:

### 4.1 First-In, First-Out (FIFO)

Assumes the **oldest** units are sold first. Ending inventory reflects the most **recent** purchase prices.

* In a period of **rising prices**: FIFO results in **lower COGS** and **higher net income** (because cheaper old units are expensed)
* Ending inventory is valued at recent (higher) prices — closer to replacement cost

### 4.2 Last-In, First-Out (LIFO)

Assumes the **newest** units are sold first. Ending inventory reflects the **oldest** purchase prices.

* In a period of **rising prices**: LIFO results in **higher COGS** and **lower net income** (because expensive new units are expensed)
* **Tax benefit**: lower taxable income means lower taxes (this is the primary motivation for LIFO)
* **LIFO is prohibited under IFRS** — only permitted under US GAAP

> **CFA Exam Tip:** The LIFO-FIFO comparison in a rising-price environment is one of the most frequently tested topics. Remember: LIFO gives lower income, lower taxes, lower inventory — and the ending inventory can be severely understated relative to current costs.

### 4.3 Weighted Average Cost

Assigns the **average cost** of all units available for sale during the period:

$$
\text{Weighted Average Cost per Unit} = \frac{\text{Cost of Goods Available for Sale}}{\text{Units Available for Sale}}
$$

Results fall between FIFO and LIFO for both COGS and ending inventory.

### Worked example

A company has the following inventory transactions:

| Date | Transaction | Units | Cost/Unit |
|------|-------------|-------|-----------|
| Jan 1 | Beginning inventory | 100 | $10 |
| Mar 15 | Purchase | 200 | $12 |
| Jul 20 | Purchase | 150 | $14 |
| Oct 5 | Purchase | 100 | $15 |

**Total units available:** 550 units. **Units sold:** 400. **Ending inventory:** 150 units.

### Why the method matters so much

The inventory method choice is arguably the **most impactful** accounting policy decision for manufacturing and retail companies. It simultaneously affects:

* **Income statement:** COGS is directly determined by the method, flowing through to gross profit, operating income, and net income.
* **Balance sheet:** Ending inventory value appears in current assets, affecting total assets, working capital, and equity.
* **Cash flow statement:** Tax payments differ because taxable income differs — this is a *real cash* effect, not just an accounting one.
* **Financial ratios:** Inventory turnover, current ratio, ROA, and ROE are all distorted by the method choice.

The table below summarises the effects in a rising-price environment (the most common case):

| Metric | FIFO | LIFO | Weighted Average |
|--------|------|------|-----------------|
| COGS | Lowest | Highest | Middle |
| Ending inventory | Highest | Lowest | Middle |
| Gross profit | Highest | Lowest | Middle |
| Net income | Highest | Lowest | Middle |
| Taxes paid | Highest | Lowest | Middle |
| Cash flow | Lowest (more tax) | Highest (less tax) | Middle |

> **Key Concept:** LIFO is the *only* method where choosing the accounting treatment that produces **lower** reported income actually **increases** cash flow (through tax savings). This is the fundamental paradox of the LIFO decision: it makes you look worse on paper but leaves you richer in reality.

In [ ]:
# ── Inventory Cost Flow Methods: FIFO, LIFO, Weighted Average ──

# Purchase schedule
purchases = [
    ('Beginning Inv.', 100, 10.0),
    ('Purchase Mar',   200, 12.0),
    ('Purchase Jul',   150, 14.0),
    ('Purchase Oct',   100, 15.0),
]

units_sold = 400
total_units = sum(qty for _, qty, _ in purchases)
units_remaining = total_units - units_sold

# Cost of goods available for sale
cogs_available = sum(qty * cost for _, qty, cost in purchases)

print(f"Units available: {total_units}")
print(f"Units sold: {units_sold}")
print(f"Units remaining: {units_remaining}")
print(f"Cost of goods available: ${cogs_available:,.2f}")
print()

# ── FIFO: sell oldest first ──
fifo_cogs = 0
fifo_remaining = units_sold
for name, qty, cost in purchases:
    taken = min(fifo_remaining, qty)
    fifo_cogs += taken * cost
    fifo_remaining -= taken
    if fifo_remaining == 0:
        break
fifo_ending_inv = cogs_available - fifo_cogs

# ── LIFO: sell newest first ──
lifo_cogs = 0
lifo_remaining = units_sold
for name, qty, cost in reversed(purchases):
    taken = min(lifo_remaining, qty)
    lifo_cogs += taken * cost
    lifo_remaining -= taken
    if lifo_remaining == 0:
        break
lifo_ending_inv = cogs_available - lifo_cogs

# ── Weighted Average ──
wa_cost_per_unit = cogs_available / total_units
wa_cogs = units_sold * wa_cost_per_unit
wa_ending_inv = units_remaining * wa_cost_per_unit

# Results
print("=" * 55)
print(f"{'Method':>20}  {'COGS':>10}  {'Ending Inv':>12}")
print("=" * 55)
print(f"{'FIFO':>20}  ${fifo_cogs:>9,.2f}  ${fifo_ending_inv:>11,.2f}")
print(f"{'LIFO':>20}  ${lifo_cogs:>9,.2f}  ${lifo_ending_inv:>11,.2f}")
print(f"{'Weighted Avg':>20}  ${wa_cogs:>9,.2f}  ${wa_ending_inv:>11,.2f}")
print("=" * 55)
print(f"\nVerification (COGS + Ending Inv = {cogs_available:,.2f} for all methods):")
for method, c, e in [('FIFO', fifo_cogs, fifo_ending_inv),
                      ('LIFO', lifo_cogs, lifo_ending_inv),
                      ('WA', wa_cogs, wa_ending_inv)]:
    print(f"  {method}: {c + e:,.2f} ✓" if np.isclose(c + e, cogs_available) else f"  {method}: MISMATCH")

As expected with rising prices:
* **FIFO** produces the **lowest COGS** (sells cheap old units) and **highest ending inventory** (retains expensive new units).
* **LIFO** produces the **highest COGS** (sells expensive new units) and **lowest ending inventory** (retains cheap old units).
* **Weighted Average** falls in between.

The total cost of goods available for sale ($6,800) is the same under all methods — the only difference is how it is *allocated* between COGS and ending inventory.

### Understanding *why* the results differ

The intuition is straightforward once you trace the physical cost layers:

**FIFO** peels costs off the top of the stack (oldest first):
- Sells: 100 units @ \$10 + 200 @ \$12 + 100 @ \$14 = \$4,900
- Remaining: 50 @ \$14 + 100 @ \$15 = \$2,200 (recent, higher prices)

**LIFO** peels costs off the bottom of the stack (newest first):
- Sells: 100 @ \$15 + 150 @ \$14 + 150 @ \$12 = \$5,400
- Remaining: 100 @ \$10 + 50 @ \$12 = \$1,600 (old, lower prices)

The \$500 difference in COGS (\$5,400 − \$4,900) is the **LIFO reserve** — a critical concept for analysts converting between methods.

> **Common Mistake:** Students sometimes think FIFO and LIFO are about the *physical movement* of goods. They are not. A grocery store using LIFO accounting still puts the oldest milk at the front of the shelf. The cost flow assumption is purely an accounting convention — it determines which *costs* are assigned to sold units, not which *physical units* are shipped.

In [ ]:
# ── Impact on financial statements (assuming 25% tax rate) ──

tax_rate = 0.25
revenue_val = 7200  # assume $18/unit * 400 units sold

methods = ['FIFO', 'LIFO', 'Weighted Avg']
cogs_vals = [fifo_cogs, lifo_cogs, wa_cogs]
ending_inv_vals = [fifo_ending_inv, lifo_ending_inv, wa_ending_inv]

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
x = np.arange(len(methods))

# COGS comparison
axes[0].bar(x, cogs_vals, color=[PRIMARY, SECONDARY, TERTIARY], edgecolor='white')
axes[0].set_xticks(x)
axes[0].set_xticklabels(methods)
axes[0].set_ylabel('$')
axes[0].set_title('COGS')
for i, v in enumerate(cogs_vals):
    axes[0].text(i, v + 30, f'${v:,.0f}', ha='center', fontsize=11, fontweight='bold')

# Ending Inventory comparison
axes[1].bar(x, ending_inv_vals, color=[PRIMARY, SECONDARY, TERTIARY], edgecolor='white')
axes[1].set_xticks(x)
axes[1].set_xticklabels(methods)
axes[1].set_title('Ending Inventory (Balance Sheet)')
for i, v in enumerate(ending_inv_vals):
    axes[1].text(i, v + 15, f'${v:,.0f}', ha='center', fontsize=11, fontweight='bold')

# Net Income comparison
ni_vals = [(revenue_val - c) * (1 - tax_rate) for c in cogs_vals]
axes[2].bar(x, ni_vals, color=[PRIMARY, SECONDARY, TERTIARY], edgecolor='white')
axes[2].set_xticks(x)
axes[2].set_xticklabels(methods)
axes[2].set_title('Net Income (after 25% tax)')
for i, v in enumerate(ni_vals):
    axes[2].text(i, v + 10, f'${v:,.0f}', ha='center', fontsize=11, fontweight='bold')

fig.suptitle('Impact of Inventory Method on Financial Statements (Rising Prices)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Tax savings
fifo_tax = (revenue_val - fifo_cogs) * tax_rate
lifo_tax = (revenue_val - lifo_cogs) * tax_rate
print(f"LIFO tax savings vs FIFO: ${fifo_tax - lifo_tax:,.2f}")
print("This is why US companies choose LIFO — real cash savings from lower taxes")

The triple bar chart makes the trade-off crystal clear. LIFO reports lower net income — which looks worse to investors — but generates real cash savings through lower taxes. This is the fundamental tension in the LIFO decision.

> **Key Concept:** The inventory method choice does not change the *physical flow* of goods. A company using LIFO does not literally sell its newest inventory first. The cost flow assumption is purely an *accounting convention* that determines how costs are allocated between the income statement and balance sheet.

### The LIFO conformity rule

In the US, companies that use LIFO for tax purposes must also use LIFO for financial reporting (the **LIFO conformity rule**). This is unusual — most accounting choices can differ between tax and financial reporting. The conformity rule means that the tax benefit comes at the cost of reporting lower earnings to shareholders.

### When prices are falling

Everything reverses when prices decline:
* FIFO produces **higher** COGS (selling expensive old units) and **lower** ending inventory.
* LIFO produces **lower** COGS (selling cheap new units) and **higher** ending inventory.

In practice, persistent deflation is rare in most industries, so the rising-price scenario dominates the LIFO discussion.

> **CFA Exam Tip:** The exam will specify whether prices are rising or falling. Do not assume. Also remember that **IFRS prohibits LIFO** — any question involving an IFRS company using LIFO is testing whether you know this restriction.

---
## 5. LIFO Reserve and LIFO Liquidation

### LIFO Reserve

The **LIFO reserve** is the difference between inventory valued at FIFO and inventory valued at LIFO:

$$
\text{LIFO Reserve} = \text{Inventory}_{FIFO} - \text{Inventory}_{LIFO}
$$

US GAAP requires companies using LIFO to disclose the LIFO reserve in footnotes. This disclosure allows analysts to **convert** LIFO financials to a FIFO basis for comparability:

$$
\text{Inventory}_{FIFO} = \text{Inventory}_{LIFO} + \text{LIFO Reserve}
$$

$$
\text{COGS}_{FIFO} = \text{COGS}_{LIFO} - \Delta\text{LIFO Reserve}
$$

> **CFA Exam Tip:** Converting LIFO to FIFO is a *core analytical skill*. The exam expects you to adjust both the balance sheet (add LIFO reserve to inventory and equity, net of tax) and the income statement (subtract the change in LIFO reserve from COGS).

### LIFO Liquidation

When a LIFO company sells more units than it purchases in a period, it **dips into** old LIFO layers at historically low costs. This produces:
* Abnormally **low COGS** → inflated gross profit
* A **one-time** earnings boost that is not sustainable

> **Common Mistake:** LIFO liquidation artificially inflates earnings. Analysts must identify and exclude the LIFO liquidation profit when assessing sustainable profitability. The LIFO reserve *decreasing* is the telltale sign.

### Balance sheet and income statement adjustments

When converting a LIFO company to a FIFO basis, analysts must adjust **both** statements:

**Balance sheet adjustments:**
1. Add LIFO reserve to inventory (increases current assets)
2. Add LIFO reserve × tax rate to deferred tax liability (increases liabilities)  
3. Add LIFO reserve × (1 − tax rate) to retained earnings (increases equity)

**Income statement adjustments:**
1. Subtract the *change* in LIFO reserve from COGS (if reserve increased, FIFO COGS is lower)
2. Recompute tax expense on the adjusted pre-tax income
3. Net income increases by the change in LIFO reserve × (1 − tax rate)

> **Key Concept:** The LIFO reserve adjustment is one of the most important analytical tools in cross-company comparison. Without it, comparing a US LIFO company with an international FIFO company (IFRS prohibits LIFO) would be misleading — the LIFO company's inventory and equity are systematically understated relative to current replacement costs.

In [ ]:
# ── LIFO Reserve and LIFO-to-FIFO conversion ──

# Suppose a LIFO company reports:
lifo_inventory = 1_500     # balance sheet (LIFO basis)
lifo_reserve_begin = 400   # from prior year footnote
lifo_reserve_end = 500     # from current year footnote
lifo_cogs_reported = 3_200 # income statement (LIFO basis)

# Convert to FIFO
fifo_inventory_adj = lifo_inventory + lifo_reserve_end
fifo_cogs_adj = lifo_cogs_reported - (lifo_reserve_end - lifo_reserve_begin)

# Tax impact of reserve
tax_rate_lr = 0.25
equity_adj = lifo_reserve_end * (1 - tax_rate_lr)

print("LIFO-to-FIFO Conversion")
print("=" * 50)
print(f"{'':>30} {'LIFO':>10} {'FIFO':>10}")
print("-" * 50)
print(f"{'Inventory':>30} ${lifo_inventory:>9,} ${fifo_inventory_adj:>9,}")
print(f"{'COGS':>30} ${lifo_cogs_reported:>9,} ${fifo_cogs_adj:>9,}")
print(f"{'Equity adjustment (net of tax)':>30} {'':>10} ${equity_adj:>9,.0f}")
print()
print(f"LIFO Reserve (end): ${lifo_reserve_end:,}")
print(f"Change in LIFO Reserve: ${lifo_reserve_end - lifo_reserve_begin:,}")
print(f"FIFO COGS is ${lifo_cogs_reported - fifo_cogs_adj:,} lower (higher profit on FIFO basis)")

The conversion reveals the magnitude of LIFO distortion: FIFO inventory is \$500 higher (reflecting more recent, higher costs), while FIFO COGS is \$100 lower (the change in LIFO reserve). The equity adjustment of \$375 represents the after-tax increase in shareholder value that would be recognised under FIFO.

### Tracking the LIFO reserve over time

The **trend** in the LIFO reserve is analytically more valuable than its level:

* **Growing LIFO reserve** → prices are rising and/or the company is building inventory. The gap between LIFO and FIFO statements is widening.
* **Stable LIFO reserve** → prices and inventory levels are roughly constant. LIFO and FIFO produce similar results.
* **Shrinking LIFO reserve** → possible **LIFO liquidation** (selling more than purchasing, dipping into old cheap layers). This inflates earnings unsustainably and is a red flag.

> **CFA Exam Tip:** If the exam states that a company's LIFO reserve *decreased*, immediately consider LIFO liquidation. Compute the tax-adjusted earnings impact: Earnings boost = $\Delta$LIFO Reserve × (1 − tax rate). This amount should be *excluded* from normalised earnings because it represents a one-time benefit from depleting old inventory layers, not a sustainable improvement in operations.

---
## 6. Depreciation Methods

Depreciation allocates the cost of a tangible long-lived asset over its **useful life**. It is a *non-cash* charge — no cash leaves the company — but it reduces reported income and the carrying value (book value) of the asset.

$$
\text{Depreciable Base} = \text{Cost} - \text{Residual (Salvage) Value}
$$

### 6.1 Straight-Line (SL)

The simplest method — equal expense each period:

$$
\text{Annual Depreciation}_{SL} = \frac{\text{Cost} - \text{Residual Value}}{\text{Useful Life}}
$$

### 6.2 Double Declining Balance (DDB)

An **accelerated** method — higher depreciation in early years:

$$
\text{Depreciation}_t = \frac{2}{\text{Useful Life}} \times \text{Book Value}_{t-1}
$$

Note: DDB does **not** subtract the residual value from the depreciable base. Instead, depreciation stops when book value reaches the residual value.

In practice, companies switch from DDB to straight-line when the SL amount exceeds the DDB amount — this ensures the asset is fully depreciated by the end of its life.

### 6.3 Units of Production

Depreciation is proportional to **actual usage**:

$$
\text{Depreciation}_t = \frac{\text{Units Produced}_t}{\text{Total Estimated Units}} \times (\text{Cost} - \text{Residual Value})
$$

> **Key Concept:** The method does not change the *total* depreciation over the asset's life (always = Cost - Residual Value). It only changes the *timing*. Accelerated methods front-load the expense, reducing early-year profits but generating tax savings (a real cash benefit).

> **CFA Exam Tip:** If the exam provides information about a company changing from accelerated to straight-line depreciation, this *increases* reported income — but it is a change in accounting *estimate*, not a genuine improvement in operations. Flag it as a potential earnings quality concern.

### Worked example

A machine costs $100,000, has a residual value of $10,000, a useful life of 5 years, and is expected to produce 50,000 units total.

### Component depreciation

Under IFRS (IAS 16), companies must depreciate **significant components** of an asset separately if they have different useful lives. For example, an aircraft might be decomposed into:
* Airframe: 25-year life
* Engines: 10-year life (replaced more frequently)
* Interior fittings: 5-year life

US GAAP permits but does not require component depreciation. This creates another IFRS vs GAAP comparability issue — IFRS companies may report higher depreciation in early years if they separately depreciate short-lived components.

> **Key Concept:** Depreciation is not optional or arbitrary — it reflects the real economic consumption of an asset's productive capacity. A company that underestimates depreciation will overstate both its current profitability and its asset values. The distortion compounds over time: assets appear inflated on the balance sheet, and when they are eventually replaced, the large capital expenditure creates a sudden cash flow shock that the income statement never warned about.

In [ ]:
# ── Depreciation Methods Comparison ──

cost = 100_000
residual = 10_000
useful_life = 5
depreciable_base = cost - residual

# Units of production schedule
units_per_year = np.array([12_000, 15_000, 10_000, 8_000, 5_000])
total_units = units_per_year.sum()

# ── Straight-Line ──
sl_depr = np.full(useful_life, depreciable_base / useful_life)
sl_bv = cost - np.cumsum(sl_depr)

# ── Double Declining Balance (with switch to SL) ──
ddb_rate = 2 / useful_life
ddb_depr = np.zeros(useful_life)
bv = cost
for t in range(useful_life):
    ddb_amount = bv * ddb_rate
    # Check if switching to SL gives more depreciation
    sl_remaining = (bv - residual) / (useful_life - t)
    if sl_remaining >= ddb_amount:
        # Switch to straight-line for remaining years
        for s in range(t, useful_life):
            ddb_depr[s] = (bv - residual) / (useful_life - t)
        break
    # Don't depreciate below residual
    ddb_depr[t] = min(ddb_amount, bv - residual)
    bv -= ddb_depr[t]
ddb_bv = cost - np.cumsum(ddb_depr)

# ── Units of Production ──
uop_depr = (units_per_year / total_units) * depreciable_base
uop_bv = cost - np.cumsum(uop_depr)

# Display schedule
years_dep = np.arange(1, useful_life + 1)
print(f"Machine: Cost=${cost:,}  Residual=${residual:,}  Life={useful_life} years")
print()
print(f"{'Year':>4}  {'SL Depr':>10}  {'SL BV':>10}  {'DDB Depr':>10}  {'DDB BV':>10}  {'UoP Depr':>10}  {'UoP BV':>10}")
print("-" * 75)
for i in range(useful_life):
    print(f"{years_dep[i]:>4}  ${sl_depr[i]:>9,.0f}  ${sl_bv[i]:>9,.0f}  "
          f"${ddb_depr[i]:>9,.0f}  ${ddb_bv[i]:>9,.0f}  ${uop_depr[i]:>9,.0f}  ${uop_bv[i]:>9,.0f}")
print("-" * 75)
print(f"Total ${sum(sl_depr):>9,.0f}  {'':>10}  ${sum(ddb_depr):>9,.0f}  {'':>10}  ${sum(uop_depr):>9,.0f}")
print(f"\nAll methods depreciate the same total: ${depreciable_base:,}")

The depreciation schedule confirms several key properties:

1. **Total depreciation is identical** across all three methods (\$90,000 = Cost − Residual). The method only affects *timing*.
2. **DDB front-loads** the expense: Year 1 depreciation (\$40,000) is more than double the straight-line amount (\$18,000). By Year 3, the DDB method switches to straight-line because the remaining SL amount exceeds the DDB calculation.
3. **Units of Production** mirrors actual usage: Year 2 has the highest depreciation because production was highest (15,000 units).

All three methods reach the same ending book value (\$10,000 = residual value) at the end of Year 5.

### Tax implications of depreciation method choice

Since depreciation is tax-deductible, higher early depreciation means lower taxable income and lower taxes in the early years. The **present value** of tax savings is higher under accelerated methods because more of the deduction comes sooner:

$$\text{PV of Tax Shield}_{DDB} > \text{PV of Tax Shield}_{SL}$$

This is why many companies use accelerated depreciation for *tax* purposes while using straight-line for *financial reporting* — creating a **deferred tax liability** that reflects the timing difference.

> **Key Concept:** The choice of depreciation method is one of the most impactful accounting policy decisions. It affects reported income, asset values, tax payments, and therefore nearly every financial ratio. When comparing companies, always check whether they use the same depreciation methods and useful life assumptions.

In [ ]:
# ── Visualise book value curves and annual depreciation ──

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Left: Book value over time
years_plot = np.arange(0, useful_life + 1)
ax1.plot(years_plot, np.insert(sl_bv, 0, cost), 'o-', color=PRIMARY, linewidth=2, label='Straight-Line')
ax1.plot(years_plot, np.insert(ddb_bv, 0, cost), 's-', color=SECONDARY, linewidth=2, label='Double Declining Balance')
ax1.plot(years_plot, np.insert(uop_bv, 0, cost), '^-', color=TERTIARY, linewidth=2, label='Units of Production')
ax1.axhline(residual, color='grey', linestyle='--', alpha=0.5, label=f'Residual (${residual:,})')
ax1.set_xlabel('Year')
ax1.set_ylabel('Book Value ($)')
ax1.set_title('Book Value Over Time')
ax1.legend()

# Right: Annual depreciation expense
x = np.arange(useful_life)
width = 0.25
ax2.bar(x - width, sl_depr, width, label='Straight-Line', color=PRIMARY)
ax2.bar(x, ddb_depr, width, label='DDB', color=SECONDARY)
ax2.bar(x + width, uop_depr, width, label='Units of Production', color=TERTIARY)
ax2.set_xticks(x)
ax2.set_xticklabels([f'Year {i+1}' for i in range(useful_life)])
ax2.set_ylabel('Depreciation Expense ($)')
ax2.set_title('Annual Depreciation Expense')
ax2.legend()

plt.tight_layout()
plt.show()

The charts illustrate the core trade-off:
* **Straight-line** produces a linear, predictable book value decline — preferred by companies wanting stable earnings.
* **DDB** front-loads depreciation — book value drops steeply in early years. This is more conservative and provides larger tax shields early.
* **Units of Production** follows actual usage — depreciation varies with production volume, which may better match the asset's actual economic consumption.

> **CFA Exam Tip:** In the early years of an asset's life, accelerated depreciation means *lower* reported income, *lower* reported assets, and therefore *lower* equity. This affects ratios like ROA and ROE. When comparing companies using different methods, analysts must adjust for depreciation policy differences.

### Choosing a depreciation method

The choice depends on the nature of the asset and management's reporting preferences:

| Method | Best suited for | Rationale |
|--------|----------------|-----------|
| Straight-line | Buildings, office furniture | Provide roughly equal benefit each period |
| DDB | Technology equipment, vehicles | Lose value rapidly in early years; matches economic depreciation |
| Units of production | Mining equipment, aircraft engines | Value consumed in proportion to usage, not time |

In practice, **straight-line dominates** — over 85% of companies use it for financial reporting, largely because it produces the smoothest earnings pattern. However, for *tax* purposes, most jurisdictions offer accelerated depreciation (or even immediate expensing under provisions like US Section 179) to incentivise capital investment.

---
## 7. Impairment of Long-Lived Assets

When a long-lived asset's value has declined below its carrying (book) value, the asset must be **written down** — an impairment charge is recognised on the income statement.

### US GAAP (ASC 360) — Two-step test

1. **Recoverability test:** Is the carrying value > undiscounted future cash flows? If yes → impairment exists.
2. **Measurement:** Impairment loss = Carrying value - Fair value.

### IFRS (IAS 36) — One-step test

An asset is impaired when its carrying value exceeds the **recoverable amount**, defined as the higher of:
* **Fair value less costs of disposal**
* **Value in use** (PV of expected future cash flows)

$$
\text{Impairment Loss} = \text{Carrying Value} - \text{Recoverable Amount}
$$

### Key difference: reversal

| | US GAAP | IFRS |
|---|---------|------|
| **Impairment reversal (non-goodwill)** | **Not permitted** | **Permitted** (up to original carrying value) |
| **Goodwill impairment reversal** | Not permitted | Not permitted |

> **Key Concept:** IFRS's allowance for impairment reversal means that write-downs are potentially reversible — creating opportunities for earnings management. An aggressive company could write down assets in a bad year (reducing the base for future depreciation) and reverse in a good year. Analysts should track impairments and reversals carefully.

> **Common Mistake:** Impairment charges are often treated as "non-recurring." But if a company takes impairments repeatedly, that signals systematic over-investment or optimistic asset valuation — a structural problem, not a one-off event.

### Impairment indicators (triggering events)

Both IFRS and US GAAP require impairment testing when there are **indicators** that an asset's value may have declined:

**External indicators:**
* Significant decline in the asset's market value
* Adverse changes in the technological, market, or economic environment
* Increase in interest rates (reduces the present value of future cash flows)
* Company's market capitalisation falls below net book value of assets

**Internal indicators:**
* Evidence of physical damage or obsolescence
* Plans to discontinue or restructure operations using the asset
* Asset's performance is worse than expected (lower output, higher maintenance costs)

> **CFA Exam Tip:** Under US GAAP, impairment is tested *only when indicators exist* (except for goodwill and indefinite-life intangibles, which are tested annually). Under IFRS, the same indicator-based approach applies for most assets, with annual testing required for goodwill, indefinite-life intangibles, and intangibles not yet available for use.

### Worked example — impairment under IFRS

A machine has:
* Carrying value: \$500,000
* Fair value less costs of disposal: \$380,000
* Value in use (PV of future cash flows): \$420,000
* Recoverable amount = max(\$380,000, \$420,000) = **\$420,000**
* Impairment loss = \$500,000 − \$420,000 = **\$80,000**

The \$80,000 is charged to the income statement, and the asset's carrying value is written down to \$420,000. Future depreciation will be based on this lower value — which actually *increases* future profitability (lower depreciation expense). This is why analysts must be wary of write-downs as a potential earnings management tool.

### Goodwill impairment — a special case

Goodwill arises from acquisitions (purchase price exceeding fair value of identifiable net assets). Unlike tangible assets:
* Goodwill is **never amortised** (under both IFRS and current US GAAP)
* It is tested for impairment **annually** and whenever indicators exist
* Impairment is **never reversed** (under both frameworks)
* Large goodwill impairments often signal that an acquisition destroyed shareholder value

> **Common Mistake:** A goodwill impairment does not mean the company lost cash — it means management is acknowledging that a past acquisition was worth less than originally recorded. The cash was spent at the time of acquisition; the impairment simply corrects the balance sheet to reflect current economic reality.

---
## 8. Working Capital and the Cash Conversion Cycle

### Working capital

$$
\text{Working Capital} = \text{Current Assets} - \text{Current Liabilities}
$$

Positive working capital means the company can cover its short-term obligations. However, *too much* working capital may indicate inefficient use of resources (excess inventory, slow collections).

### The cash conversion cycle (CCC)

The CCC measures how many days it takes to convert a dollar of investment in inventory into a dollar of cash from sales:

$$
\text{CCC} = \text{DIO} + \text{DSO} - \text{DPO}
$$

where:

$$
\text{DIO (Days Inventory Outstanding)} = \frac{\text{Inventory}}{\text{COGS}} \times 365
$$

$$
\text{DSO (Days Sales Outstanding)} = \frac{\text{Accounts Receivable}}{\text{Revenue}} \times 365
$$

$$
\text{DPO (Days Payable Outstanding)} = \frac{\text{Accounts Payable}}{\text{COGS}} \times 365
$$

> **Key Concept:** A **shorter CCC is better** — it means the company converts its investments into cash faster. A negative CCC (common in retail, e.g. Amazon, Dell) means the company collects cash from customers *before* it pays suppliers — effectively using supplier financing to fund operations.

### Economic interpretation

The CCC can be visualised as a timeline:

```
Day 0: Purchase inventory (cash out to supplier → starts DPO clock)
Day DIO: Sell the inventory (cash not yet received → starts DSO clock)
Day DIO + DSO: Collect from customer (cash in)
Day DPO: Pay the supplier (cash out)

CCC = (DIO + DSO) - DPO = gap between paying supplier and collecting from customer
```

If CCC > 0: the company must finance this gap (through working capital loans, retained cash, etc.)
If CCC < 0: the company gets paid *before* it pays — suppliers are financing its operations.

### Working capital management — the balancing act

Working capital management is a constant tension between **liquidity** and **efficiency**:

* **Too much working capital** (high inventory, generous credit terms) ties up capital that could be invested elsewhere. It signals inefficiency.
* **Too little working capital** (lean inventory, aggressive collections, delayed supplier payments) risks stockouts, customer dissatisfaction, and strained supplier relationships.

The optimal level depends on the industry, the company's bargaining power, and its access to short-term financing. Companies with strong credit lines can afford to run leaner because they have a backup source of liquidity.

### Negative working capital — is it bad?

Counterintuitively, negative working capital is not always a sign of financial distress. Many successful companies operate with negative working capital by design:

* **Grocery retailers** (Walmart, Costco) sell inventory quickly for cash and pay suppliers on extended terms.
* **Subscription businesses** (SaaS companies) collect annual payments upfront (deferred revenue is a current liability) and deliver the service over 12 months.
* **Insurance companies** collect premiums before paying claims, creating a permanent "float."

In these cases, negative working capital is a *structural advantage*, not a weakness — the business model generates cash before it needs to be spent.

> **CFA Exam Tip:** When the exam presents negative working capital, check the *reason*. If it is driven by high deferred revenue or high accounts payable with fast inventory turns, it may be a sign of operational efficiency. If it is driven by low cash and high short-term debt, it is likely a liquidity concern.

In [ ]:
# ── Cash Conversion Cycle Computation ──

# MfgCo financial data
inv_data = {
    'Revenue': 500_000,
    'COGS': 320_000,
    'Avg Inventory': 45_000,
    'Avg Accounts Receivable': 60_000,
    'Avg Accounts Payable': 38_000,
}

dio = (inv_data['Avg Inventory'] / inv_data['COGS']) * 365
dso = (inv_data['Avg Accounts Receivable'] / inv_data['Revenue']) * 365
dpo = (inv_data['Avg Accounts Payable'] / inv_data['COGS']) * 365
ccc = dio + dso - dpo

print("Cash Conversion Cycle — MfgCo")
print("=" * 45)
print(f"DIO (Days Inventory Outstanding):  {dio:>6.1f} days")
print(f"DSO (Days Sales Outstanding):      {dso:>6.1f} days")
print(f"DPO (Days Payable Outstanding):    {dpo:>6.1f} days")
print(f"                                   {'─' * 10}")
print(f"CCC = DIO + DSO - DPO:             {ccc:>6.1f} days")
print()
print(f"Interpretation: MfgCo must finance {ccc:.0f} days of operations")
print(f"between paying suppliers and collecting from customers.")

MfgCo's CCC of approximately 51 days means that on average, 51 days elapse between the point when cash goes out (paying suppliers) and when it comes back in (collecting from customers). During this gap, the company must finance its operations through working capital — either from retained cash, bank credit lines, or other sources.

### Decomposing the CCC

Each component tells a different part of the working capital story:

* **DIO (51 days):** Inventory sits in the warehouse for ~51 days before being sold. Lower is generally better (less capital tied up), but cutting DIO too aggressively can lead to stockouts.
* **DSO (44 days):** Customers take ~44 days to pay after a sale. This reflects the company's credit terms and collection efficiency. Rising DSO may signal deteriorating receivables quality.
* **DPO (43 days):** The company takes ~43 days to pay its own suppliers. Longer DPO preserves cash but may strain supplier relationships or forfeit early-payment discounts.

> **Common Mistake:** Students sometimes try to minimise each component independently. But the CCC is a *system* — extending DPO (paying suppliers later) reduces CCC just as effectively as reducing DIO (selling inventory faster). The optimal strategy depends on the relative costs and negotiating power with suppliers vs customers.

In [ ]:
# ── CCC timeline visualisation ──

fig, ax = plt.subplots(figsize=(12, 4))

# Timeline bars
ax.barh('Inventory\n(DIO)', dio, left=0, height=0.3, color=PRIMARY, alpha=0.8)
ax.barh('Receivables\n(DSO)', dso, left=dio, height=0.3, color=TERTIARY, alpha=0.8)
ax.barh('Payables\n(DPO)', dpo, left=0, height=0.3, color=SECONDARY, alpha=0.8)

# CCC bracket
ccc_start = dpo
ccc_end = dio + dso
ax.annotate('', xy=(ccc_end, -0.5), xytext=(ccc_start, -0.5),
            arrowprops=dict(arrowstyle='<->', color='black', lw=2))
ax.text((ccc_start + ccc_end) / 2, -0.7, f'CCC = {ccc:.0f} days',
        ha='center', fontsize=12, fontweight='bold')

# Labels on bars
ax.text(dio/2, 0, f'DIO = {dio:.0f}d', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(dio + dso/2, 0, f'DSO = {dso:.0f}d', ha='center', va='center', fontsize=10, color='white', fontweight='bold')
ax.text(dpo/2, 0, f'DPO = {dpo:.0f}d', ha='center', va='center', fontsize=10, color='white', fontweight='bold')

ax.set_xlabel('Days')
ax.set_title('Cash Conversion Cycle Timeline')
ax.set_xlim(-5, dio + dso + 10)
ax.set_ylim(-1.2, 0.8)
ax.axvline(0, color='grey', linewidth=0.5, linestyle='--')
plt.tight_layout()
plt.show()

The timeline chart makes the CCC intuitive: the company buys inventory (blue bar), then waits for the customer to pay (green bar), while its own payment to suppliers (red bar) runs in parallel. The CCC is the gap between when cash goes out (end of DPO) and when it comes back in (end of DSO).

> **CFA Exam Tip:** When comparing companies, a rising CCC over time suggests deteriorating working capital management — inventory is piling up, customers are paying slower, or the company is paying suppliers faster. Any of these trends warrants investigation.

In [ ]:
# ── Multi-company CCC comparison ──

companies = {
    'MfgCo (Industrial)':    {'DIO': dio, 'DSO': dso, 'DPO': dpo},
    'RetailFast (Grocery)':  {'DIO': 15, 'DSO': 3, 'DPO': 35},
    'SoftwareCo (SaaS)':     {'DIO': 0, 'DSO': 55, 'DPO': 25},
    'LuxuryGoods (Fashion)': {'DIO': 120, 'DSO': 45, 'DPO': 60},
}

fig, ax = plt.subplots(figsize=(10, 5))
y_pos = np.arange(len(companies))
ccc_vals = []

for name, vals in companies.items():
    ccc_val = vals['DIO'] + vals['DSO'] - vals['DPO']
    ccc_vals.append(ccc_val)

colors_ccc = [PRIMARY if v >= 0 else TERTIARY for v in ccc_vals]
ax.barh(y_pos, ccc_vals, color=colors_ccc, edgecolor='white', height=0.5)
ax.set_yticks(y_pos)
ax.set_yticklabels(list(companies.keys()))
ax.set_xlabel('Cash Conversion Cycle (days)')
ax.set_title('CCC Comparison Across Industries')
ax.axvline(0, color='black', linewidth=0.8)
for i, v in enumerate(ccc_vals):
    ax.text(v + 2 if v >= 0 else v - 8, i, f'{v:.0f}d', va='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()

print("Industry patterns:")
for (name, vals), ccc_val in zip(companies.items(), ccc_vals):
    print(f"  {name}: CCC = {ccc_val:.0f} days (DIO={vals['DIO']:.0f} DSO={vals['DSO']:.0f} DPO={vals['DPO']:.0f})")

The comparison reveals how dramatically CCC varies by industry:
* **Grocery retail** has a *negative* CCC — they sell perishable goods quickly (low DIO), customers pay immediately (low DSO), and they negotiate long payment terms with suppliers (high DPO).
* **Luxury fashion** has the longest CCC — high-value inventory sits for months, and customers may use credit.
* **SaaS** has zero inventory but waits ~2 months for enterprise customers to pay invoices.

> **Key Concept:** There is no "right" CCC — it depends on the industry and business model. What matters is the *trend* and how a company compares to *peers in the same industry*.

### Working capital as a source of competitive advantage

Companies with negative CCCs effectively have **free financing from their supply chain**. Amazon is the canonical example — it collects payment from customers immediately (DPO ≈ 0 for consumers) but pays suppliers on 60-90 day terms. This generates billions in "float" that Amazon can invest in growth before any of its own capital is required.

> **Key Concept:** When analysing an acquisition target, the CCC tells you how much *working capital investment* will be required as the business grows. A company with a CCC of 90 days needs significantly more capital to fund growth than one with a CCC of 30 days — even if their revenue growth rates are identical. This is a frequently overlooked factor in valuation.

### Dell's pioneering negative CCC

In the 1990s, Dell Computers revolutionised manufacturing by achieving a negative CCC through:
* **Build-to-order model:** No finished goods inventory (DIO near zero)
* **Direct sales:** Customers paid at time of order (DSO near zero)
* **Supplier terms:** Components paid on 60+ day terms (high DPO)

The result: Dell was collecting cash from customers *weeks* before it paid suppliers for the components. This generated enormous free cash flow that funded growth without external financing — a structural advantage that competitors building to stock could not easily replicate.

---
## 9. Liquidity Ratios

Liquidity ratios measure a company's ability to meet **short-term obligations**. They are computed from balance sheet data and answer: *"Can this company pay its bills?"*

### Common liquidity ratios

| Ratio | Formula | Interpretation |
|-------|---------|---------------|
| **Current Ratio** | $\frac{\text{Current Assets}}{\text{Current Liabilities}}$ | Broadest liquidity measure; includes all current assets |
| **Quick Ratio** | $\frac{\text{Cash + ST Investments + Receivables}}{\text{Current Liabilities}}$ | Excludes inventory (may be hard to sell quickly) |
| **Cash Ratio** | $\frac{\text{Cash + ST Investments}}{\text{Current Liabilities}}$ | Most conservative; only truly liquid assets |
| **Defensive Interval** | $\frac{\text{Cash + ST Inv + Receivables}}{\text{Daily Operating Expenses}}$ | Days the company can operate from liquid assets alone |

> **Key Concept:** A current ratio > 1 does not guarantee safety. If most current assets are slow-moving inventory, the company may still face a cash crunch. That is why analysts look at the quick ratio and cash ratio alongside the current ratio.

> **Common Mistake:** Very high liquidity ratios are not necessarily good. A current ratio of 5.0 may indicate that the company is hoarding cash rather than investing in growth — a potential sign of poor capital allocation.

### The defensive interval ratio

The defensive interval ratio deserves special attention because it measures liquidity in *time* rather than as a ratio:

$$\text{Defensive Interval} = \frac{\text{Cash + ST Investments + Receivables}}{\text{Daily Operating Expenses}}$$

where Daily Operating Expenses = (COGS + SG&A − D&A) / 365. Depreciation is excluded because it is non-cash.

A defensive interval of 90 days means the company can fund 90 days of operations from its most liquid assets without any new revenue — a much more intuitive measure than saying "the quick ratio is 1.4."

> **Common Mistake:** Don't confuse liquidity with solvency. A company can have excellent liquidity ratios (plenty of cash to meet short-term obligations) while being technically insolvent (total liabilities exceed total assets). Liquidity measures the ability to meet obligations in the *short term*; solvency measures long-term financial viability. Both perspectives are needed.

In [ ]:
# ── Liquidity Ratios for three companies ──

balance_sheets = {
    'MfgCo': {
        'Cash': 25_000, 'ST_Investments': 10_000, 'Receivables': 60_000,
        'Inventory': 45_000, 'Prepaid': 5_000,
        'Current_Liab': 85_000, 'Daily_OpEx': 1_200
    },
    'RetailFast': {
        'Cash': 15_000, 'ST_Investments': 0, 'Receivables': 5_000,
        'Inventory': 30_000, 'Prepaid': 2_000,
        'Current_Liab': 60_000, 'Daily_OpEx': 900
    },
    'SoftwareCo': {
        'Cash': 80_000, 'ST_Investments': 20_000, 'Receivables': 40_000,
        'Inventory': 0, 'Prepaid': 3_000,
        'Current_Liab': 35_000, 'Daily_OpEx': 500
    }
}

print(f"{'Ratio':>22}  {'MfgCo':>8}  {'Retail':>8}  {'Software':>8}")
print("─" * 50)

for ratio_name in ['Current', 'Quick', 'Cash', 'Defensive Interval']:
    vals = []
    for company, bs in balance_sheets.items():
        ca = bs['Cash'] + bs['ST_Investments'] + bs['Receivables'] + bs['Inventory'] + bs['Prepaid']
        quick_assets = bs['Cash'] + bs['ST_Investments'] + bs['Receivables']
        cash_assets = bs['Cash'] + bs['ST_Investments']

        if ratio_name == 'Current':
            vals.append(ca / bs['Current_Liab'])
        elif ratio_name == 'Quick':
            vals.append(quick_assets / bs['Current_Liab'])
        elif ratio_name == 'Cash':
            vals.append(cash_assets / bs['Current_Liab'])
        elif ratio_name == 'Defensive Interval':
            vals.append(quick_assets / bs['Daily_OpEx'])

    if ratio_name == 'Defensive Interval':
        print(f"{ratio_name + ' (days)':>22}  {vals[0]:>7.0f}d  {vals[1]:>7.0f}d  {vals[2]:>7.0f}d")
    else:
        print(f"{ratio_name:>22}  {vals[0]:>8.2f}  {vals[1]:>8.2f}  {vals[2]:>8.2f}")

The ratios tell very different stories depending on which measure you use:
* **MfgCo** has a decent current ratio (1.71) but much of that is inventory — the quick ratio (1.12) is substantially lower.
* **RetailFast** has a current ratio below 1.0 — typical for grocery retailers who rely on rapid inventory turnover and supplier credit.
* **SoftwareCo** has the highest ratios across the board — cash-rich with no inventory, typical of mature SaaS companies.

> **CFA Exam Tip:** Always compute multiple liquidity ratios rather than relying on just one. The *gap* between the current ratio and quick ratio reveals how much liquidity depends on inventory.

### Interpreting the defensive interval

The defensive interval ratio answers a more practical question than the other liquidity ratios: *"How many days can this company survive without any new cash inflows?"* It is particularly relevant for:

* **Distressed companies** — how long can they hold out while restructuring?
* **Startups** — how many months of "runway" remain before the next funding round?
* **Pandemic/crisis scenarios** — how long can a company weather a sudden revenue stoppage?

### Trend analysis matters more than levels

A single-period liquidity ratio is a snapshot — useful but limited. The real diagnostic power comes from tracking ratios **over time**:

* **Declining current ratio** with **stable quick ratio** → inventory is being run down (could be efficiency improvement or demand weakness).
* **Declining quick ratio** with **stable current ratio** → receivables are being collected faster (or written off) while inventory builds.
* **All ratios declining together** → the company is systematically becoming less liquid, which may signal financial distress.

> **CFA Exam Tip:** The exam often presents 3-5 years of ratio data and asks you to identify the trend and its implications. Practice reading ratio tables quickly — focus on direction and magnitude of change, not just the absolute levels.

---
## 10. Off-Balance-Sheet Items

Certain economic obligations may not appear on the balance sheet, creating a potentially misleading picture of the company's financial position:

### 10.1 Operating leases (pre-IFRS 16 / ASC 842)

Before the new lease standards, operating leases were disclosed only in footnotes — the asset and liability were *off balance sheet*. This meant companies could operate with significant asset commitments that did not appear as debt.

**Post-reform:** IFRS 16 (2019) and ASC 842 (2019) now require *most* leases to be recognised on the balance sheet as a right-of-use asset and a lease liability. However:
* IFRS 16: *all* leases on balance sheet (single model) — except short-term (<12 months) and low-value
* ASC 842: two models — **finance leases** (similar to IFRS) and **operating leases** (on balance sheet but with different expense pattern)

### 10.2 Special Purpose Entities (SPEs / VIEs)

Entities created for a specific purpose (securitisation, project finance) that may be structured to avoid consolidation. Post-Enron reforms (FIN 46R / IFRS 10) tightened the rules, requiring consolidation when the sponsoring company bears the majority of risks and rewards.

### 10.3 Contingent liabilities

Obligations that depend on a future event:
* **Probable and estimable:** Recognised on the balance sheet (GAAP) or "more likely than not" (IFRS)
* **Probable but not estimable:** Disclosed in footnotes only
* **Remote:** No recognition or disclosure required

> **Key Concept:** Off-balance-sheet items are the "dark matter" of financial analysis. The footnotes to financial statements contain critical information about these items. Professional analysts read footnotes as carefully as the primary statements.

> **CFA Exam Tip:** The exam may ask you to *capitalise* an operating lease by computing the present value of minimum lease payments and adding it as both an asset and a liability. This adjusts ratios like debt-to-equity and asset turnover — often dramatically.

### 10.4 Pension obligations

Defined benefit pension plans create complex off-balance-sheet dynamics:

* The **projected benefit obligation (PBO)** — the present value of all future pension payments owed to current and former employees — can be enormous.
* The **plan assets** — investments set aside to fund these payments — may or may not cover the PBO.
* The **funded status** (plan assets minus PBO) appears on the balance sheet, but the components (discount rate assumptions, expected returns, actuarial gains/losses) are disclosed in footnotes.

Companies can significantly affect reported pension expense through **assumption changes**:
* A higher discount rate reduces the PBO → less pension expense → higher profit
* A higher expected return on plan assets reduces pension expense → higher profit

> **CFA Exam Tip:** The exam tests your ability to recognise how pension assumption changes affect financial statements. A company that raises its discount rate by 50 basis points can reduce its pension liability by billions — a purely accounting change with no operational significance.

### 10.5 Analytical framework for off-balance-sheet items

When encountering any off-balance-sheet item, apply this framework:

1. **Identify** the item from footnote disclosures
2. **Estimate** the economic obligation or asset value
3. **Adjust** the balance sheet by capitalising the item (add both an asset and a liability)
4. **Recalculate** key ratios (debt/equity, asset turnover, ROA) using the adjusted figures
5. **Assess materiality** — does the adjustment meaningfully change the analytical conclusions?

> **Key Concept:** The trend in accounting standards is toward *more* on-balance-sheet recognition (IFRS 16 leases, IFRS 9 expected credit losses). But significant off-balance-sheet items will always exist because financial statements can only capture obligations that meet recognition criteria — many economic exposures (e.g., pending litigation, environmental liabilities) remain uncertain enough to be disclosed rather than recognised.

---
## 11. Summary & References

This notebook covered the essential balance sheet and working capital topics:

* The **classified balance sheet** groups items by liquidity and maturity, enabling quick assessment of financial position.
* **Inventory cost flow methods** (FIFO, LIFO, weighted average) significantly affect both COGS and ending inventory — and therefore net income and balance sheet values.
* The **LIFO reserve** allows conversion between methods; **LIFO liquidation** creates artificial profit when old layers are depleted.
* **Depreciation methods** (straight-line, DDB, units of production) differ in timing but not total expense — accelerated methods are more conservative.
* **Impairment** testing differs between IFRS (one-step, reversible) and US GAAP (two-step, irreversible).
* The **cash conversion cycle** (DIO + DSO - DPO) measures working capital efficiency and varies dramatically by industry.
* **Liquidity ratios** (current, quick, cash) assess short-term solvency; always use multiple ratios together.
* **Off-balance-sheet items** (leases, SPEs, contingencies) can hide significant obligations; footnotes are essential reading.

The next notebook covers the **cash flow statement** — constructing the indirect method, computing free cash flow, and analysing cash flow quality.

---

### References

1. CFA Institute. *CFA Program Curriculum Level I*, "Understanding Balance Sheets" and "Inventories."
2. IAS 2 — *Inventories*.
3. IAS 16 — *Property, Plant and Equipment*.
4. IAS 36 — *Impairment of Assets*.
5. IFRS 16 — *Leases*.
6. Stickney, C. & Weil, R. *Financial Accounting: An Introduction to Concepts, Methods and Uses*.
7. Palepu, K. & Healy, P. *Business Analysis and Valuation*, 6th ed.

### Key formulas to remember

| Formula | Application |
|---------|-------------|
| FIFO ending inv. = LIFO ending inv. + LIFO Reserve | Converting between inventory methods |
| FIFO COGS = LIFO COGS − $\Delta$LIFO Reserve | Income statement adjustment |
| SL Depreciation = (Cost − Residual) / Life | Straight-line annual charge |
| DDB Depreciation = (2/Life) × Book Value | Accelerated depreciation |
| DIO = (Inventory / COGS) × 365 | Days inventory outstanding |
| DSO = (Receivables / Revenue) × 365 | Days sales outstanding |
| DPO = (Payables / COGS) × 365 | Days payable outstanding |
| CCC = DIO + DSO − DPO | Cash conversion cycle |
| Current Ratio = CA / CL | Broadest liquidity measure |
| Quick Ratio = (Cash + ST Inv + Recv) / CL | Excludes inventory |

### CFA exam preparation notes

Balance sheet and working capital topics represent approximately 10-15% of FSA questions on the CFA Level 1 exam. The highest-frequency topics are:

1. **FIFO vs LIFO** effects on COGS, inventory, net income, and taxes (rising vs falling prices)
2. **LIFO reserve** adjustments to convert LIFO to FIFO for comparability
3. **Depreciation method** effects on income, assets, and ratios
4. **Impairment** testing differences between IFRS and US GAAP
5. **Cash conversion cycle** computation and interpretation
6. **Liquidity ratios** — current, quick, and cash — and what each reveals
7. **Off-balance-sheet items** — especially operating lease capitalisation